# 🎓 LAB 02 – Value Iteration & Policy Iteration trong MDP rời rạc

**Môi trường:** `CliffWalking-v0` · `Taxi-v3` — **Thư viện:** Gymnasium, NumPy

**Sinh viên thực hiện:** *<điền họ tên>* · **MSSV:** *<điền MSSV>*

Notebook giải quyết trọn vẹn yêu cầu của Lab 02:

1. Cài đặt **Value Iteration** (PART 1) và **Policy Iteration** (PART 2) từ nguyên lý, kèm comment lý giải từng bước.
2. In kết quả chạy trên cả hai môi trường, kèm **thí nghiệm mở rộng** về `γ` và phần thưởng phạt.
3. Thảo luận phân tích: reward/discount định hình hành vi; vì sao VI và PI hội tụ về cùng policy tối ưu.
4. Trả lời tất cả các câu hỏi lý luận: **THINK FIRST**, **THINK BEFORE CODING**, **REFLECT**.

> **Ghi chú phiên bản:** đề dùng `CliffWalking-v0` và `Taxi-v3`. Trên gymnasium ≥ 1.1, hai tên này bị đánh dấu deprecated và `gym.make` báo lỗi `DeprecatedEnv`, thay bằng `CliffWalking-v1` / `Taxi-v4`. Động lực học (bảng chuyển tiếp `P`, reward) **giống hệt nhau**, chỉ khác chuỗi phiên bản. Notebook dùng `try/except` để chạy được trên cả bản gymnasium cũ lẫn mới, và kiểm chứng lại dynamics ngay ở phần khám phá bảng `P`.

## 🎯 Mục tiêu học tập

1. Hiểu **phương trình Bellman** là nền tảng của planning trong Markov Decision Processes (MDP).
2. Cài đặt **Value Iteration** và **Policy Iteration** từ đầu (first principles).
3. Quan sát hành vi hội tụ và phân tích cấu trúc môi trường ảnh hưởng thế nào đến việc học policy tối ưu.
4. Lý giải về **hiệu quả thuật toán** chứ không chỉ viết code chạy được.

# 🧩 PART 1 – Value Iteration trên CliffWalking-v0

## 1. Giới thiệu

**CliffWalking** là gridworld 4 hàng × 12 cột (48 state, đánh số theo hàng, từ trái sang phải, bắt đầu từ hàng trên cùng):

```
hàng 0:  0  1  2  3  4  5  6  7  8  9 10 11
hàng 1: 12 13 14 15 16 17 18 19 20 21 22 23
hàng 2: 24 25 26 27 28 29 30 31 32 33 34 35
hàng 3: 36 37 38 39 40 41 42 43 44 45 46 47
         S  █  █  █  █  █  █  █  █  █  █  G
```

- **S (state 36)** là điểm xuất phát — góc **dưới-trái**; **G (state 47)** là đích — góc **dưới-phải**; **█ (state 37–46)** là vách cliff nằm **dọc hàng dưới** giữa hai ô này. (Mô tả trong đề — "start góc trên-trái, đích dưới-phải" — là cách vẽ quy ước trong sách; bản cài đặt thật của Gymnasium đặt start dưới-trái, notebook vẽ đúng theo bản cài đặt thật.)
- **4 actions:** `0=UP ↑`, `1=RIGHT →`, `2=DOWN ↓`, `3=LEFT ←`; đi vào tường thì đứng nguyên tại chỗ.
- **Reward:** mỗi bước **−1**; bước vào ô cliff **−100** và bị **đưa về điểm xuất phát** (episode chưa kết thúc); bước vào G thì episode **kết thúc** (`done=True`).

**Mục tiêu:** 👉 cài đặt **Value Iteration** để tính policy tối ưu né cliff.

In [1]:
import gymnasium as gym
import numpy as np

# Đề dùng "CliffWalking-v0"; gymnasium >= 1.1 thay bằng v1 (dynamics giống hệt).
try:
    env = gym.make("CliffWalking-v0")
except gym.error.DeprecatedEnv:
    env = gym.make("CliffWalking-v1")

n_states = env.observation_space.n
n_actions = env.action_space.n
print("States:", n_states, "Actions:", n_actions)

States: 48 Actions: 4


C:\Users\ADMIN\AppData\Roaming\Python\Python314\site-packages\gymnasium\envs\registration.py:513: DeprecationWarning: WARN: The environment CliffWalking-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


In [2]:
# Toàn bộ động lực học của MDP nằm trong bảng P của môi trường "thô" (unwrapped):
#     P[s][a] = list các (xác_suất p, next_state, reward, done)
# Với mỗi cặp (state, action), bảng này cho biết agent có thể rơi vào những
# next_state nào, với xác suất bao nhiêu, nhận reward bao nhiêu, và episode
# có kết thúc tại đó hay không. Value Iteration chỉ cần đúng bảng này.
P = env.unwrapped.P

print("P[36][1] - từ start (3,0) đi RIGHT -> rơi xuống cliff:", P[36][1])
print("P[35][2] - từ (2,11) đi DOWN -> bước vào đích       :", P[35][2])
print("P[24][0] - từ (2,0) đi UP -> lên (1,0)              :", P[24][0])

# Kiểm chứng: môi trường này có stochastic không?
deterministic = all(
    len(P[s][a]) == 1 and P[s][a][0][0] == 1.0
    for s in range(n_states) for a in range(n_actions)
)
print()
print("Mọi cặp (s,a) đều có đúng 1 transition với p = 1.0 (deterministic)?", deterministic)

P[36][1] - từ start (3,0) đi RIGHT -> rơi xuống cliff: [(1.0, np.int64(36), -100, False)]
P[35][2] - từ (2,11) đi DOWN -> bước vào đích       : [(1.0, np.int64(47), -1, True)]
P[24][0] - từ (2,0) đi UP -> lên (1,0)              : [(1.0, np.int64(12), -1, False)]

Mọi cặp (s,a) đều có đúng 1 transition với p = 1.0 (deterministic)? True


## 3. Task Instructions — 🧠 THINK FIRST

*(Đề yêu cầu trả lời trong comment — các khối comment lớn nằm ngay trong hàm `value_iteration` bên dưới; phần này trình bày đầy đủ hơn.)*

**Câu 1 — Phương trình Bellman *optimality* cho bài toán này?**

$$V^*(s) \;=\; \max_a Q^*(s,a) \;=\; \max_a \sum_{s'} p(s'\mid s,a)\,\Big[\, r(s,a,s') + \gamma\, V^*(s') \,\Big]$$

với quy ước chuẩn $V^*(s') = 0$ khi $s'$ là state kết thúc (bước vào đích, `done=True`) — vì hết episode, không còn tương lai để chiết khấu. Riêng với CliffWalking, mỗi cặp $(s,a)$ chỉ có một transition ($p=1$, đã kiểm chứng ở trên) nên tổng quy về: $V^*(s) = \max_a \big[\,r(s,a) + \gamma V^*(s')\,\big]$, trong đó $r = -100$ nếu $s'$ là ô cliff, $r = -1$ nếu là bước thường, và phần $V^*(s')$ bị bỏ đi nếu $s'$ là đích.

**Câu 2 — `max_a Q(s,a)` nghĩa intuitively là gì?**

Là **giá trị kỳ vọng tốt nhất** có thể đạt được nếu đang đứng ở $s$, được **chọn hành động tốt nhất ngay lúc này** và hành xử tối ưu ở mọi bước sau. Nó "chấm điểm" từng hành động bằng (reward tức thời + giá trị tương lai chiết khấu) rồi lấy phương án sáng giá nhất — và chính vì thế $\max_a Q(s,a) \equiv V^*(s)$. Đây là cây cầu nối giữa "giá trị của một state" và "giá trị của một cặp (state, action)": muốn biết state tốt tới đâu, phải biết action tốt nhất ở state đó mang lại gì.

**Câu 3 — Vì sao có trade-off giữa `γ` và tốc độ hội tụ?**

Toán tử Bellman tối ưu $\mathcal{T}$ là một **phép co (contraction) với hệ số co đúng bằng $\gamma$**: $\|\mathcal{T}V - \mathcal{T}U\|_\infty \le \gamma \,\|V - U\|_\infty$. Nghĩa là mỗi vòng lặp chỉ đảm bảo thu hẹp sai số tối đa $\gamma$ lần:

- `γ` **nhỏ** → co mạnh mỗi vòng → hội tụ sau ít vòng lặp; nhưng agent **cận thị** vì tương lai bị "phai" theo $\gamma^k$ rất nhanh, có thể bỏ qua các chuỗi reward dài hạn tốt hơn.
- `γ → 1` → nhìn được xa, đánh giá dài hạn đúng hơn; nhưng mỗi vòng chỉ co rất ít nên **số vòng cần thiết tăng xấp xỉ theo $1/(1-\gamma)$**, và bound sai số $\theta\gamma/(1-\gamma)$ cũng nới lỏng ra.

Trade-off cốt lõi: **chất lượng tối ưu dài hạn đổi lấy tốc độ hội tụ** — và ngược lại.

## 🧩 IMPLEMENTATION — thiết kế hàm `value_iteration`

Sơ đồ thuật toán (tự suy luận từ phương trình Bellman, không sao chép cấu trúc có sẵn):

1. Khởi tạo $V_0 = \mathbf{0}$.
2. **Một sweep:** với mọi state $s$, tính $q(s,a) = \sum_{s'} p(s'\mid s,a)\,[\,r + \gamma V_{\text{cũ}}(s')\,]$ cho từng action (quy tắc: `done=True` → phần tương lai bằng 0), rồi đặt $V_{\text{mới}}(s) = \max_a q(s,a)$.
3. Đo `delta = max_s |V_mới(s) − V_cũ(s)|`; lặp lại bước 2 cho tới khi `delta < theta`.
4. **Trích policy:** $\pi(s) = \arg\max_a q(s,a)$ dùng $V = V^*$ vừa tìm được.

Hàm nhận thêm tham số tuỳ chọn `stats` (một dict) chỉ để **ghi lại số sweep** phục vụ phần đo đạc/so sánh với Policy Iteration ở PART 2 — không ảnh hưởng gì đến thuật toán.

In [3]:
import time


def value_iteration(env, gamma=0.9, theta=1e-6, stats=None):
    '''
    Tính value function tối ưu và policy tối ưu bằng Value Iteration.

    Ta phải:
    - Suy ra quy tắc cập nhật cho V(s)                -> TODO 1
    - Áp dụng cho tới khi hội tụ (delta < theta)      -> TODO 1 + TODO 3
    - Trích policy tối ưu từ V(s)                     -> TODO 2
    '''
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    # Bảng động lực học: P[s][a] = list của (p, next_state, reward, done)
    P = env.unwrapped.P

    V = np.zeros(n_states)

    # =====================================================================
    # TODO 1: Vòng lặp cập nhật chính cho V(s)
    #
    # P, R, gamma ở đây "đại diện" cho gì?
    #   * P[s][a] là TOÀN BỘ động lực học: từ (s, a) có thể tới những state nào,
    #     với xác suất p bao nhiêu, reward tức thời r là bao nhiêu, và episode
    #     có dừng tại đó không (done).
    #   * R không phải một bảng riêng lẻ — reward nằm TRONG từng transition,
    #     nên expected return phải là tổng CÓ TRỌNG SỐ theo p trên các transition.
    #   * gamma chiết khấu tương lai: giá trị của state kế tiếp được nhân gamma,
    #     thể hiện "điểm thưởng ở tương lai ít giá trị hơn điểm thưởng hiện tại".
    #
    # Quy tắc cập nhật (Bellman optimality backup):
    #     V_new(s) = max_a  sum_{s'} p(s'|s,a) * [ r(s,a,s') + gamma * V_old(s') ]
    # Điểm then chốt: khi done=True (bước vào đích, hết episode) thì KHÔNG cộng
    # thêm V(s') — tương đương quy ước V(terminal) = 0. Nếu quên bước này, giá
    # trị sẽ tự bootstrap qua chính nó (V = r + gamma*V) và hội tụ về một giá
    # trị sai hoàn toàn (ví dụ mọi state trong CliffWalking sẽ dính về -10).
    # Ta quét toàn bộ state kiểu "synchronous": vế phải chỉ dùng V cũ.
    # =====================================================================
    sweeps = 0
    while True:
        delta = 0.0
        V_new = np.zeros(n_states)
        for s in range(n_states):
            action_values = np.zeros(n_actions)
            for a in range(n_actions):
                expected = 0.0
                for prob, next_state, reward, done in P[s][a]:
                    expected += prob * (reward + gamma * (0.0 if done else V[next_state]))
                action_values[a] = expected
            V_new[s] = np.max(action_values)          # max_a Q(s,a)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        sweeps += 1
        if delta < theta:
            break

    # =====================================================================
    # TODO 2: Sau khi hội tụ, trích greedy policy:
    #     policy(s) = argmax_a  sum_{s'} p(s'|s,a) * [ r + gamma * V*(s') ]
    # Vì V đã (gần) đúng V*, argmax này chính là policy tối ưu. Nếu nhiều
    # action đồng giá trị (tie), np.argmax chọn action có index nhỏ nhất.
    # =====================================================================
    policy = np.zeros(n_states, dtype=int)
    for s in range(n_states):
        action_values = np.zeros(n_actions)
        for a in range(n_actions):
            expected = 0.0
            for prob, next_state, reward, done in P[s][a]:
                expected += prob * (reward + gamma * (0.0 if done else V[next_state]))
            action_values[a] = expected
        policy[s] = int(np.argmax(action_values))

    # =====================================================================
    # TODO 3: Stopping criterion hoạt động thế nào?
    #
    # Sau mỗi sweep ta đo delta = max_s |V_new(s) - V(s)|: "cơn địa chấn" lớn
    # nhất mà vòng sweep vừa gây ra trên toàn bộ state. Toán tử Bellman là
    # phép co với hệ số gamma nên khoảng cách tới V* cũng co theo: khi
    # delta < theta thì sai số thật bị chặn bởi  theta * gamma / (1 - gamma).
    # Nói cách khác, khi KHÔNG còn state nào thay đổi đáng kể sau một lượt
    # backup nữa thì thêm vòng lặp cũng không đổi gì đáng kể -> dừng là an
    # toàn và có bound sai số rõ ràng.
    # (Đặc thù môi trường deterministic như CliffWalking: giá trị lan truyền
    # từng ô như một "làn sóng" xuất phát từ đích; khi làn sóng phủ hết lưới,
    # mọi V(s) đạt NGUYÊN V* cùng lúc và delta tụt về ~0 — xem số sweep ở
    # phần Evaluation: 15 sweep = khoảng cách xa nhất tới đích (14) + 1.)
    # =====================================================================
    if stats is not None:
        stats["sweeps"] = sweeps

    return V, policy

## 4. Evaluation

Đề yêu cầu: **(1)** in final average state value; **(2)** hiển thị policy bằng mũi tên `↑ ↓ ← →`; **(3)** thảo luận bằng văn bản (nằm ở các mục 🔍 bên dưới sau khi có số liệu).

In [4]:
# Công cụ hiển thị: lưới policy bằng mũi tên và lưới giá trị V(s)
ARROWS = {0: "↑", 1: "→", 2: "↓", 3: "←"}   # 0=UP, 1=RIGHT, 2=DOWN, 3=LEFT
N_COLS = 12


def render_cliff_policy(policy, ncols=N_COLS):
    """Lưới policy 4x12; S = start (36), G = goal (47), █ = cliff (37-46)."""
    lines = []
    for r in range(4):
        row = []
        for c in range(ncols):
            s = r * ncols + c
            if r == 3 and c == 0:
                row.append("S")
            elif r == 3 and c == 11:
                row.append("G")
            elif r == 3 and 1 <= c <= 10:
                row.append("█")
            else:
                row.append(ARROWS[int(policy[s])])
        lines.append(" ".join(row))
    return "\n".join(lines)


def render_cliff_values(V, ncols=N_COLS):
    """Lưới V(s) làm tròn 2 chữ số, cùng bố cục với render_cliff_policy."""
    lines = []
    for r in range(4):
        row = [f"{V[r * ncols + c]:6.2f}" for c in range(ncols)]
        lines.append(" ".join(row))
    return "\n".join(lines)


stats_cliff = {}
t0 = time.perf_counter()
V_cliff, policy_cliff = value_iteration(env, gamma=0.9, theta=1e-6, stats=stats_cliff)
elapsed = time.perf_counter() - t0

print("Số sweep để hội tụ :", stats_cliff["sweeps"])
print("Thời gian chạy     : %.4f giây" % elapsed)
print()
print("1) Final average state value: %.6f" % np.mean(V_cliff))
print("   (V[start=36] = %.4f | V min = %.4f | V max = %.4f)"
      % (V_cliff[36], V_cliff.min(), V_cliff.max()))
print()
print("2) Lưới giá trị V(s):")
print(render_cliff_values(V_cliff))
print()
print("   Lưới policy tối ưu (↑↓←→):")
print(render_cliff_policy(policy_cliff))

Số sweep để hội tụ : 15
Thời gian chạy     : 0.0067 giây

1) Final average state value: -5.088570
   (V[start=36] = -7.4581 | V min = -7.7123 | V max = -1.0000)

2) Lưới giá trị V(s):
 -7.71  -7.46  -7.18  -6.86  -6.51  -6.13  -5.70  -5.22  -4.69  -4.10  -3.44  -2.71
 -7.46  -7.18  -6.86  -6.51  -6.13  -5.70  -5.22  -4.69  -4.10  -3.44  -2.71  -1.90
 -7.18  -6.86  -6.51  -6.13  -5.70  -5.22  -4.69  -4.10  -3.44  -2.71  -1.90  -1.00
 -7.46  -7.18  -6.86  -6.51  -6.13  -5.70  -5.22  -4.69  -4.10  -3.44  -1.00  -1.00

   Lưới policy tối ưu (↑↓←→):
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
S █ █ █ █ █ █ █ █ █ █ G


### 🔍 Thảo luận (3): pattern quan sát được gần khu vực cliff

- Policy tối ưu **men ngay theo mép cliff**: từ start đi `↑` một bước lên hàng 2, chạy `→` dọc **hàng nằm ngay phía trên vách cliff**, rồi `↓` xuống đích ở cột phải. Đây là đường ngắn nhất an toàn: khoảng cách Manhattan từ start tới đích là 11 bước, nhưng hàng dưới bị vách chặn nên bắt buộc phải vòng thêm đúng 2 bước (lên 1, xuống 1) → **13 bước**.
- Các ô **ngay trên vách cliff** vẫn chỉ `→`, không `↑` chạy trốn: trong môi trường **deterministic** này, đứng cạnh cliff không tốn gì — chỉ có *bước vào* mới bị phạt −100. Nên "đi men mép" là tối ưu. (Đây đúng là hành vi *optimal-but-risky* mà Q-learning thể hiện trong ví dụ kinh điển của Sutton & Barto; các thuật toán học bằng sampling có exploration như SARSA với $\varepsilon$-greedy mới chọn đường "an toàn" chạy ở hàng trên.)
- Ở hàng 0 và hàng 1, `→` và `↓` **hoàn toàn đồng giá trị**: đi ngang trước rồi xuống, hay xuống trước rồi đi ngang, tốn đúng bằng nhau số bước. `np.argmax` phá tie bằng cách chọn index nhỏ hơn (RIGHT = 1 < DOWN = 2) nên ta thấy `→` chiếm toàn bộ hai hàng này.
- Giá trị $V(s)$ giảm dần theo khoảng cách tới đích: từ $-1.0$ tại ô (2,11) (một bước vào đích, phần tương lai bằng 0 do `done=True`) tới $\approx -7.71$ ở hàng trên cùng (14 bước, chiết khấu $\gamma=0.9$).

## 🧪 Thí nghiệm mở rộng 1 — chính là câu thảo luận "γ = 0.9 so với 0.99?"

Đề hỏi: *policy nhạy cảm thế nào với các giá trị `γ` khác nhau (0.9 so với 0.99)?* Thay vì trả lời cảm tính, ta chạy thật Value Iteration với `γ ∈ {0.5, 0.9, 0.99}` rồi đối chiếu policy và giá trị.

In [5]:
gammas = [0.5, 0.9, 0.99]
gamma_results = {}

for g in gammas:
    st = {}
    Vg, polg = value_iteration(env, gamma=g, theta=1e-6, stats=st)
    gamma_results[g] = (Vg, polg, st["sweeps"])
    print(f"=== gamma = {g} | sweeps = {st['sweeps']} | mean V = {np.mean(Vg):.4f} ===")
    print(render_cliff_policy(polg))
    print()

print("gamma | sweeps | mean V   | số ô policy khác so với gamma=0.9")
for g in gammas:
    diffs = int(np.sum(gamma_results[g][1] != gamma_results[0.9][1]))
    print(f"{g:5.2f} | {gamma_results[g][2]:6d} | {np.mean(gamma_results[g][0]):8.4f} | {diffs}")

=== gamma = 0.5 | sweeps = 15 | mean V = -1.8802 ===
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
S █ █ █ █ █ █ █ █ █ █ G

=== gamma = 0.9 | sweeps = 15 | mean V = -5.0886 ===
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
S █ █ █ █ █ █ █ █ █ █ G

=== gamma = 0.99 | sweeps = 15 | mean V = -7.1408 ===
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
S █ █ █ █ █ █ █ █ █ █ G

gamma | sweeps | mean V   | số ô policy khác so với gamma=0.9
 0.50 |     15 |  -1.8802 | 0
 0.90 |     15 |  -5.0886 | 0
 0.99 |     15 |  -7.1408 | 0


### 💬 Thảo luận: policy có nhạy cảm với γ không?

**Kết quả đo được:** policy tối ưu **giống hệt nhau 100%** (0 ô khác nhau) ở cả `γ = 0.5, 0.9, 0.99`, và số sweep hội tụ cũng bằng nhau (15). Thứ duy nhất thay đổi là **giá trị**: mean V lần lượt $-1.88 \to -5.09 \to -7.14$ khi γ tăng.

**Vì sao policy không đổi?** Đường men mép cliff (13 bước) là đường **ngắn nhất**, và mọi phương án khác đều tệ hơn ở **mọi** mức $\gamma \in (0,1)$: phần phạt rơi cliff (−100, kèm bị reset về start) áp đảo so với chi phí thêm ~2 bước (−1/bước). Discount chỉ "co" giá trị của các chuỗi dài, không đổi **thứ tự** giữa các phương án khi khoảng cách quá lớn.

**Vì sao giá trị đổi?** $V$ của một state cách đích $d$ bước xấp xỉ $-\sum_{k=1}^{d}\gamma^k$ — tổng hình học. γ nhỏ làm các phần tử ở xa "phai" nhanh, nên state càng xa đích mất giá càng mạnh: mean V giảm từ −7.14 (γ = 0.99, nhìn dài hạn) xuống −1.88 (γ = 0.5, cận thị).

**Khi nào policy MỚI nhạy cảm với γ?** Khi phương án "ngắn hơn nhưng rủi ro hơn" và phương án "dài hơn nhưng an toàn" gần bằng nhau — lúc đó γ quyết định mức coi trọng tương lai. Ở đây hai bên chênh nhau quá xa (−100 so với −2), nên policy rất **robust** với γ. Mức độ nhạy cảm thật sự nằm ở **phần thưởng phạt**, được kiểm chứng ở thí nghiệm 2 ngay sau đây.

## 🧪 Thí nghiệm mở rộng 2 — "Nếu reward rơi cliff đổi từ −100 thành −10 thì sao?"

Đề chỉ yêu cầu **thảo luận**, nhưng ta có thể trả lời chắc chắn bằng thực nghiệm: deep-copy môi trường, **sửa trực tiếp bảng P** (thay reward của mọi transition đang là −100 bằng giá trị mới), rồi chạy lại Value Iteration — planning thuần túy cho phép làm điều này vì toàn bộ động lực học nằm trong `P`.

In [6]:
import copy


def cliff_env_with_penalty(penalty):
    """Bản sao CliffWalking với reward rơi cliff = penalty (mặc định môi trường: -100)."""
    em = copy.deepcopy(env.unwrapped)   # env "thô" chứa bảng P, sao deepcopy để không hỏng env gốc
    for s in range(n_states):
        for a in range(n_actions):
            em.P[s][a] = [
                (p, ns, (penalty if r == -100.0 else r), d)
                for (p, ns, r, d) in em.P[s][a]
            ]
    return em


# (a) Trường hợp đề hỏi: -100 -> -10
env_pen10 = cliff_env_with_penalty(-10.0)
V_pen10, pol_pen10 = value_iteration(env_pen10, gamma=0.9)
print("=== Cliff reward = -10 (gamma = 0.9) ===")
print("Mean V: %.6f (bản gốc: %.6f)" % (np.mean(V_pen10), np.mean(V_cliff)))
print("Số ô policy khác so với bản gốc (-100):", int(np.sum(pol_pen10 != policy_cliff)))
print(render_cliff_policy(pol_pen10))

# (b) Quét rộng: penalty thấp đến đâu thì policy mới THẬT SỰ thay đổi?
print()
print("penalty | số ô policy khác | V[start]  | mean V")
for pen in [-100.0, -10.0, -1.0, -0.8, -0.75, -0.7, -0.5]:
    Vp, polp = value_iteration(cliff_env_with_penalty(pen), gamma=0.9)
    print(f"{pen:7.2f} | {int(np.sum(polp != policy_cliff)):16d} | {Vp[36]:9.4f} | {np.mean(Vp):.4f}")

=== Cliff reward = -10 (gamma = 0.9) ===
Mean V: -5.088570 (bản gốc: -5.088570)
Số ô policy khác so với bản gốc (-100): 0
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
S █ █ █ █ █ █ █ █ █ █ G

penalty | số ô policy khác | V[start]  | mean V
-100.00 |                0 |   -7.4581 | -5.0886
 -10.00 |                0 |   -7.4581 | -5.0886
  -1.00 |                0 |   -7.4581 | -5.0886
  -0.80 |                0 |   -7.4581 | -5.0886
  -0.75 |                0 |   -7.4581 | -5.0886
  -0.70 |                2 |   -7.0000 | -5.0754


  -0.50 |               22 |   -5.0000 | -4.4790


### 💬 Thảo luận: đổi reward rơi cliff từ −100 thành −10 thì điều gì xảy ra?

**Câu trả lời đo được:** policy **không đổi một ô nào** và mean V giữ nguyên $-5.0886$. Bảng quét penalty chỉ ra ngưỡng thay đổi nằm rất sâu: đến $-0.75$ policy vẫn nguyên vẹn; ở $-0.70$ chỉ 2 ô lật; tới $-0.50$ thì 22 ô lật và $V(\text{start}) = -5.0 = \dfrac{-0.5}{1-\gamma}$.

**Giải thích.** Điều khiến agent né cliff **không phải con số −100**, mà là việc **bị reset về start**: rơi khỏi cliff thì phải đi lại từ đầu (≥ 13 bước nữa). Xét phương án "nhảy xuống cliff ngay từ start": return thu được là $P + \gamma V^*(\text{start})$, trong khi đi vòng an toàn được $V^*(\text{start}) = -7.458$. Nhảy chỉ bớt tệ khi

$$|P| \;<\; (1-\gamma)\cdot|V^*(\text{start})| \;=\; 0.1 \times 7.458 \;\approx\; 0.746,$$

đúng khớp ngưỡng đo được ($-0.75$ vẫn giữ, $-0.70$ đã lật). Nói cách khác:

- **−100 hay −10 đều "đắt" hơn ngưỡng rất xa → policy và giá trị không đổi một li** (đây là câu trả lời cho câu hỏi của đề: thay đổi này vô hại, nhưng cũng vô dụng — nó không tác động gì tới policy tối ưu).
- Nếu reward phạt **nhẹ hơn cả chi phí một bước vòng** (ví dụ −0.5), agent tối ưu sẽ… **nhảy xuống cliff có chủ đích rồi lặp lại mãi** — $V(\text{start}) = P/(1-\gamma)$ chính là giá trị của chu trình "nhảy → về start → nhảy". Đây là một dạng *reward hacking*.

**Bài học rút ra:** reward không chỉ "khuyến khích – trừng phạt", nó **định nghĩa lại bài toán**. Thuật toán planner hoàn toàn đúng vẫn trả về hành vi ngoài ý muốn nếu thang đo reward bị thiết kế sai.

# 🧩 PART 2 – Policy Iteration trên Taxi-v3

## 1. Giới thiệu

**Taxi** là gridworld 5×5 với **500 state** = 25 vị trí taxi × 5 trạng thái hành khách (4 điểm đón **R, G, Y, B** hoặc "đang trên xe") × 4 điểm đến. **6 actions:** `0=south ↓`, `1=north ↑`, `2=east →`, `3=west ←`, `4=pickup`, `5=dropoff`.

- **Reward:** mỗi action **−1**; pickup/dropoff **sai chỗ −10**; dropoff **đúng chỗ +20** và episode kết thúc. *(Đề nêu −1 và −10; bản cài đặt Gymnasium còn có +20 khi giao hàng thành công — kiểm chứng ở cell dưới.)*
- Chuyển tiếp **deterministic**.
- Mã hoá state: `row = s // 100`, `col = (s // 20) % 5`, `passenger = (s // 4) % 5`, `destination = s % 4`.

**Mục tiêu:** 👉 cài đặt **Policy Iteration** tìm policy lái xe tối ưu. Phần này **không có pseudocode** — toàn bộ logic do mình suy luận.

In [7]:
# Đề dùng "Taxi-v3"; gymnasium >= 1.3 thay bằng v4 (dynamics giống hệt).
try:
    env_taxi = gym.make("Taxi-v3")
except gym.error.DeprecatedEnv:
    env_taxi = gym.make("Taxi-v4")

n_states_taxi = env_taxi.observation_space.n
n_actions_taxi = env_taxi.action_space.n
print("Taxi states:", n_states_taxi, "Actions:", n_actions_taxi)

P_taxi = env_taxi.unwrapped.P
print("P[0][0]  - south từ state 0                              :", P_taxi[0][0])
print("P[0][4]  - pickup ĐÚNG chỗ (khách R(0,0), taxi (0,0))    :", P_taxi[0][4])
print("P[4][4]  - pickup SAI chỗ (khách G, taxi (0,0))          :", P_taxi[4][4])
print("P[16][5] - dropoff ĐÚNG chỗ -> +20, kết thúc episode     :", P_taxi[16][5])

n_done = sum(
    1
    for s in range(n_states_taxi)
    for a in range(n_actions_taxi)
    for (p, ns, r, d) in P_taxi[s][a]
    if d
)
print("Số transition kết thúc episode (dropoff thành công):", n_done)

Taxi states: 500 Actions: 6
P[0][0]  - south từ state 0                              : [(1.0, 100, -1, False)]
P[0][4]  - pickup ĐÚNG chỗ (khách R(0,0), taxi (0,0))    : [(1.0, 16, -1, False)]
P[4][4]  - pickup SAI chỗ (khách G, taxi (0,0))          : [(1.0, 4, -10, False)]
P[16][5] - dropoff ĐÚNG chỗ -> +20, kết thúc episode     : [(1.0, 0, 20, True)]
Số transition kết thúc episode (dropoff thành công): 4


C:\Users\ADMIN\AppData\Roaming\Python\Python314\site-packages\gymnasium\envs\registration.py:513: DeprecationWarning: WARN: The environment Taxi-v3 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


## 3. Task Instructions — 🧠 THINK BEFORE CODING

*(Trả lời trong comment ngay trong hàm `policy_iteration`; phần này trình bày đầy đủ.)*

**Câu 1 — Vì sao tách riêng bước Policy Evaluation và Policy Improvement?**

Vì đây là hai bài toán có cấu trúc khác hẳn nhau, tách ra thì mỗi bước đều đơn giản và có lời giải chắc chắn:

- **Evaluation** là bài toán *dự báo*: cho trước policy cố định $\pi$, tìm $V^\pi$ — giải hệ Bellman **tuyến tính** (không có `max`).
- **Improvement** là bài toán *tối ưu*: có $V^\pi$ rồi, chọn $\pi'(s) = \arg\max_a \sum_{s'} p\,[r + \gamma V^\pi(s')]$ — chỉ là một phép so sánh.

Luân phiên hai bước này chính là ý tưởng **Generalized Policy Iteration (GPI)**. Sức mạnh của cấu trúc nằm ở **Policy Improvement Theorem**: improvement theo greedy *không bao giờ* làm policy tệ hơn ($V^{\pi'} \ge V^\pi$ tại mọi state). Nhờ đảm bảo đó, ta không cần tối ưu tổng thể một lần — cứ "đánh giá → cải thiện → đánh giá…" là tiến đều về optimum. Tách riêng còn cho phép đánh đổi hiệu năng (dừng evaluation sớm → *modified policy iteration*).

**Câu 2 — Vì sao bước Policy Evaluation phải *lặp*, kể cả khi policy đã cố định?**

Vì $V^\pi$ là nghiệm của hệ $|S|$ phương trình tuyến tính liên hoàn:

$$V^\pi(s) = \sum_{s'} p(s'\mid s,\pi(s))\,[\,r + \gamma V^\pi(s')\,] \quad \forall s,$$

giá trị một state **phụ thuộc giá trị các state khác** — không thể tính "một phát" đúng ngay. Có thể giải trực tiếp bằng đại số tuyến tính nhưng tốn $O(|S|^3)$ và khó mở rộng. Cách lặp (iterative policy evaluation) chỉ lặp lại phép backup $V \leftarrow \mathcal{T}^\pi V$: đây là phép co hệ số $\gamma$ nên **hội tụ chắc chắn** tới $V^\pi$ duy nhất, mỗi vòng rẻ, và còn cho phép dừng sớm khi `delta` nhỏ (sai số bị chặn bởi $\theta\gamma/(1-\gamma)$ như ở PART 1).

**Câu 3 — Trong điều kiện nào policy sẽ ngừng thay đổi?**

Khi một vòng improvement **không đổi action ở bất kỳ state nào** — tức $\pi$ đã là điểm bất động của phép improvement: $\pi = \text{greedy}(V^\pi)$. Khi đó $V^\pi$ thỏa cả phương trình Bellman *tối ưu*, suy ra $\pi = \pi^*$. Quá trình này dừng sau **hữu hạn** vòng vì: số policy trong MDP hữu hạn là hữu hạn ($|A|^{|S|}$), và mỗi vòng improvement hoặc tăng nghiêm ngặt $V^\pi$, hoặc giữ nguyên ở mọi state (chính là lúc `stable = True`).

## 🧩 IMPLEMENTATION — thiết kế hàm `policy_iteration`

Sơ đồ thuật toán (tự suy luận):

1. Khởi tạo **policy ngẫu nhiên** (seed cố định để tái lập kết quả) và $V = 0$.
2. **Policy Evaluation:** lặp backup theo action $\pi(s)$ hiện tại cho tới khi `delta < theta` — thu được (xấp xỉ) $V^\pi$.
3. **Policy Improvement:** với mỗi state, chọn action greedy theo $V$; nếu có state nào đổi action → đánh dấu `stable = False`.
4. Lặp bước 2–3 cho tới khi `stable = True`; khi đó policy chính là $\pi^*$.

`V` được **dùng lại giữa các vòng** (warm start): sau khi policy chỉ đổi ít, evaluation khởi động từ $V^\pi$ cũ nên hội tụ rất nhanh.

In [8]:
def policy_iteration(env, gamma=0.9, theta=1e-6, stats=None):
    """
    Cài đặt Policy Iteration cho môi trường Taxi-v3.

    Các bước suy luận (logic tự viết, không pseudocode có sẵn):
    1. Khởi tạo policy ngẫu nhiên.
    2. Đánh giá policy bằng cách lặp cập nhật V(s).
    3. Cải thiện policy theo greedy.
    4. Lặp lại cho tới khi policy ổn định (stable).
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    P = env.unwrapped.P

    # Seed cố định để chạy lại notebook cho kết quả y hệt (đề không bắt buộc,
    # nhưng random policy khởi tạo là phần duy nhất không all-deterministic).
    np.random.seed(42)
    policy = np.random.choice(n_actions, n_states)
    V = np.zeros(n_states)
    stable = False

    outer_rounds = 0     # số vòng "evaluation + improvement" hoàn chỉnh
    eval_sweeps = 0      # tổng số sweep Bellman đã bỏ ra cho phần evaluation

    while not stable:
        # ================================================================
        # TODO 1: POLICY EVALUATION
        #
        # Với policy ĐỊNH, V^pi phải thỏa:
        #     V(s) = sum_{s'} p(s'|s, pi(s)) * [ r + gamma * V(s') ]
        # — lưu ý KHÔNG có max: hành động đã bị policy chọn hộ.
        # Không thể tính đúng ngay lần đầu vì giá trị mỗi state phụ thuộc
        # giá trị các state khác (hệ |S| phương trình liên hoàn); giải ma
        # trận trực tiếp tốn O(|S|^3). Thay vào đó ta LẶP backup: đây là
        # phép co hệ số gamma nên hội tụ chắc chắn tới V^pi duy nhất, và
        # dừng khi delta < theta (sai số chặn bởi theta*gamma/(1-gamma)).
        # Cũng xử lý done=True như PART 1: hết episode thì tương lai = 0.
        # ================================================================
        while True:
            delta = 0.0
            V_new = np.zeros(n_states)
            for s in range(n_states):
                a = policy[s]
                expected = 0.0
                for prob, next_state, reward, done in P[s][a]:
                    expected += prob * (reward + gamma * (0.0 if done else V[next_state]))
                V_new[s] = expected
                delta = max(delta, abs(V_new[s] - V[s]))
            V = V_new
            eval_sweeps += 1
            if delta < theta:
                break

        # ================================================================
        # TODO 2: POLICY IMPROVEMENT
        #
        # Với V^pi vừa đánh giá, chọn greedy action cho từng state:
        #     pi'(s) = argmax_a  sum_{s'} p(s'|s,a) * [ r + gamma * V^pi(s') ]
        # Theo Policy Improvement Theorem, pi' không thể tệ hơn pi. Nếu có
        # BẤT KỲ state nào đổi action -> policy chưa ổn định, phải đánh
        # giá lại từ đầu (stable = False) và lặp.
        # ================================================================
        stable = True
        for s in range(n_states):
            action_values = np.zeros(n_actions)
            for a in range(n_actions):
                expected = 0.0
                for prob, next_state, reward, done in P[s][a]:
                    expected += prob * (reward + gamma * (0.0 if done else V[next_state]))
                action_values[a] = expected
            best = int(np.argmax(action_values))
            if best != policy[s]:
                stable = False          # best action thay đổi -> chưa ổn định
                policy[s] = best
        outer_rounds += 1

        # ================================================================
        # TODO 3: Vì sao vòng lặp này hội tụ?
        #
        # (i)  Mỗi vòng improvement KHÔNG BAO GIỜ làm V^pi giảm (Policy
        #      Improvement Theorem), và tăng nghiêm ngặt ở mọi state mà
        #      action bị đổi.
        # (ii) Số policy của MDP hữu hạn là hữu hạn (n_actions^|S| = 6^500),
        #      không policy nào lặp lại hai lần vì V^pi tăng nghiêm ngặt.
        # (iii) Hữu hạn policy + dãy V^pi tăng đơn điệu có chặn trên V* =>
        #      dãy phải dừng; và nó chỉ dừng khi improvement giữ nguyên
        #      MỌI state, tức pi = greedy(V^pi) => V^pi thỏa Bellman tối ưu
        #      => pi chính là pi* — policy tối ưu mà Value Iteration cũng
        #      phải tìm ra (cùng nghiệm duy nhất, xem thảo luận tổng kết).
        # ================================================================

    if stats is not None:
        stats["outer"] = outer_rounds
        stats["eval_sweeps"] = eval_sweeps

    return V, policy

## 4. Testing and Analysis

Đề yêu cầu chạy **cả hai thuật toán trên cùng một môi trường** (Taxi-v3) rồi so sánh.

In [9]:
# Đúng như snippet trong đề:
V_vi, policy_vi = value_iteration(env_taxi)
V_pi, policy_pi = policy_iteration(env_taxi)

print("Mean Value (VI):", np.mean(V_vi))
print("Mean Value (PI):", np.mean(V_pi))
print("Different actions:", np.sum(policy_vi != policy_pi))

# --- Mở rộng: đo chi phí hội tụ của từng thuật toán để phục vụ REFLECT ---
st_vi, st_pi = {}, {}
t0 = time.perf_counter()
V_vi, policy_vi = value_iteration(env_taxi, gamma=0.9, theta=1e-6, stats=st_vi)
t_vi = time.perf_counter() - t0
t0 = time.perf_counter()
V_pi, policy_pi = policy_iteration(env_taxi, gamma=0.9, theta=1e-6, stats=st_pi)
t_pi = time.perf_counter() - t0

vi_backups = st_vi["sweeps"] * n_actions_taxi                 # mỗi sweep: nA backup/state
pi_backups = st_pi["eval_sweeps"] + st_pi["outer"] * n_actions_taxi  # eval: 1 backup/state + improve: nA/state

print()
print("Chi phí hội tụ trên Taxi-v3 (gamma=0.9, theta=1e-6)")
print("-" * 58)
print(f"{'Tiêu chí':34s}{'Value Iter.':>12s}{'Policy Iter.':>13s}")
print("-" * 58)
print(f"{'Thời gian (giây)':34s}{t_vi:>12.4f}{t_pi:>13.4f}")
print(f"{'Số vòng lặp chính':34s}{st_vi['sweeps']:>12d}{st_pi['outer']:>13d}")
print(f"{'Tổng sweep Bellman (evaluation)':34s}{st_vi['sweeps']:>12d}{st_pi['eval_sweeps']:>13d}")
print(f"{'~Số backup action / mỗi state':34s}{vi_backups:>12d}{pi_backups:>13d}")
print(f"{'Mean V':34s}{np.mean(V_vi):>12.4f}{np.mean(V_pi):>13.4f}")
print(f"{'Max |V_VI - V_PI|':34s}{np.abs(V_vi - V_pi).max():>12.2e}{0.0:>13.2e}")
print(f"{'Số action khác nhau':34s}{int(np.sum(policy_vi != policy_pi)):>12d}{0:>13d}")

Mean Value (VI): 2.4679209766162074
Mean Value (PI): 2.4679209766162074
Different actions: 0



Chi phí hội tụ trên Taxi-v3 (gamma=0.9, theta=1e-6)
----------------------------------------------------------
Tiêu chí                           Value Iter. Policy Iter.
----------------------------------------------------------
Thời gian (giây)                        0.1459       0.4685
Số vòng lặp chính                           19           17
Tổng sweep Bellman (evaluation)             19          369
~Số backup action / mỗi state              114          471
Mean V                                  2.4679       2.4679
Max |V_VI - V_PI|                     0.00e+00     0.00e+00
Số action khác nhau                          0            0


In [10]:
# Minh chứng policy tối ưu "chạy được": cho taxi chạy thật 2 episode
ACTIONS_TAXI = ["south", "north", "east", "west", "pickup", "dropoff"]
LOCS = {0: "R(0,0)", 1: "G(0,4)", 2: "Y(4,0)", 3: "B(4,3)", 4: "trên xe"}


def decode_taxi(s):
    return {"row": s // 100, "col": (s // 20) % 5, "pass": (s // 4) % 5, "dest": s % 4}


for seed in (17, 42):
    state, _ = env_taxi.reset(seed=seed)
    d = decode_taxi(state)
    print(f"--- seed {seed}: taxi=({d['row']},{d['col']}) "
          f"hành khách ở {LOCS[d['pass']]}, điểm đến {LOCS[d['dest']]}")
    total, steps, traj = 0.0, 0, []
    while True:
        a = int(policy_pi[state])
        next_state, reward, terminated, truncated, _ = env_taxi.step(a)
        total += reward
        steps += 1
        traj.append(f"{ACTIONS_TAXI[a]}({int(reward)})")
        state = next_state
        if terminated or truncated or steps > 60:
            break
    print("    Lộ trình:", " -> ".join(traj))
    print(f"    Tổng reward: {total:+.0f} | số bước: {steps} | kết thúc đúng: {terminated}")

--- seed 17: taxi=(4,1) hành khách ở R(0,0), điểm đến Y(4,0)
    Lộ trình: north(-1) -> north(-1) -> north(-1) -> north(-1) -> west(-1) -> pickup(-1) -> south(-1) -> south(-1) -> south(-1) -> south(-1) -> dropoff(20)
    Tổng reward: +10 | số bước: 11 | kết thúc đúng: True
--- seed 42: taxi=(3,4) hành khách ở G(0,4), điểm đến Y(4,0)
    Lộ trình: north(-1) -> north(-1) -> north(-1) -> pickup(-1) -> south(-1) -> south(-1) -> west(-1) -> west(-1) -> west(-1) -> west(-1) -> south(-1) -> south(-1) -> dropoff(20)
    Tổng reward: +8 | số bước: 13 | kết thúc đúng: True


## 🧠 REFLECT

**1. Thuật toán nào hội tụ nhanh hơn? Vì sao?**

Trên Taxi-v3 với thiết lập này ($\gamma=0.9$, $\theta=10^{-6}$, evaluation chạy tới ngưỡng), **Value Iteration nhanh hơn rõ rệt**: số liệu trong bảng cho thấy VI dùng ~114 lượt backup action trên mỗi state (19 sweep × 6 action), trong khi PI dùng ~471 (369 sweep evaluation + 17 vòng improvement × 6), và thời gian chạy chênh lệch cùng chiều (~3–4 lần). Ba lý do:

- **Evaluation của PI quá "sạch máu":** chạy tới `theta = 1e-6` nghĩa là mỗi vòng phải đánh giá gần chính xác $V^\pi$, trong khi phần lớn độ chính xác ấy **bị vứt bỏ** ngay ở bước improvement (chỉ cần thứ tự các action đúng là đủ, không cần giá trị chính xác).
- **Khởi tạo random** đẩy PI qua tới 17 vòng evaluation–improvement mới ổn định.
- VI với phép backup `max` lan truyền thông tin qua **mọi action** mỗi sweep, và Taxi là môi trường deterministic nên — giống CliffWalking — giá trị lan như "làn sóng" từ các state kết thúc và hội tụ chính xác chỉ sau 19 sweep (~bằng độ sâu tối đa của không gian state).

**Lưu ý quan trọng:** kết luận này phụ thuộc thiết lập. Với `gamma → 1`, số sweep của VI bùng nổ theo $1/(1-\gamma)$ trong khi số vòng của PI gần như không đổi — PI (đặc biệt bản *modified* với evaluation rút gọn) thường thắng ở γ lớn; và với policy khởi tạo tốt, PI cần rất ít vòng.

**2. Kích thước môi trường và tính stochastic ảnh hưởng thế nào đến hội tụ?**

- **Kích thước:** mỗi sweep tốn $O(|S|\,|A|\,\bar{t})$ với $\bar{t}$ là số transition trung bình mỗi cặp $(s,a)$. Taxi (500 × 6) nặng hơn Cliff (48 × 4) cả về số state lẫn số action, nên một sweep đắt hơn — nhưng đổi lại Taxi có `P` thưa (deterministic, $\bar t = 1$).
- **Stochasticity:** khi một cặp $(s,a)$ rải xác suất ra nhiều next_state, mỗi backup nặng thêm, và quan trọng hơn: **hiện tượng "làn sóng phủ hết → delta ≈ 0" mất đi** — sai số giờ giảm theo đúng hệ số co $\gamma$ mỗi sweep. Với $\gamma = 0.99$ trên môi trường stochastic, cần cỡ $\ln\theta/\ln\gamma \approx 1{,}400$ sweep, so với 15 sweep deterministic ở CliffWalking.

**3. Bước nào chiếm chi phí tính toán: evaluation hay improvement?**

**Evaluation, áp đảo:** PI trên Taxi dành 369/386 lượt sweep (~95%) cho evaluation, improvement chỉ 17 vòng — mỗi vòng chỉ một lượt quét chọn `argmax`. Điều này nhất quán với lý thuyết: evaluation phải *giải gần trọn một hệ phương trình* mỗi vòng, improvement chỉ *so sánh* các phương án. Đây cũng chính là động cơ của **modified policy iteration**: đánh giá lười vài chục sweep là đủ để improvement vẫn chọn đúng action, đổi chi phí evaluation lấy số vòng_outer nhiều hơn mà tổng công giảm mạnh.

# 📌 Thảo luận tổng hợp

## (a) Reward và discount factor định hình hành vi thế nào?

Thí nghiệm ở PART 1 cho thấy ba tầng tác động:

- **Thang reward quyết định cấu trúc bài toán.** Phạt rơi cliff −100 so với −1 mỗi bước làm mọi policy tối ưu phải né vách — và vì rơi còn bị *reset về start*, ngưỡng thật sự của hành vi nằm ở $(1-\gamma)|V^*(\text{start})| \approx 0.75$, không phải ở −100: đổi −100 → −10 không đổi gì, nhưng hạ xuống −0.5 thì agent chủ động nhảy cliff (reward hacking). Kết luận: **hành vi tối ưu phản ánh tỷ lệ giữa các mức reward, không phải giá trị tuyệt đối của chúng.**
- **Discount quyết định "tầm nhìn".** $\gamma$ nhỏ → agent trọng phần thưởng gần; $\gamma \to 1$ → coi trọng tương lai. Trong CliffWalking policy bất biến với $\gamma \in \{0.5, 0.9, 0.99\}$ vì chênh lệch giữa các phương án quá lớn, nhưng **giá trị** thay đổi mạnh (mean V từ −1.88 đến −7.14). Khi các phương án gần bằng nhau, chính $\gamma$ là biến quyết định giữa "đường ngắn rủi ro" và "đường dài an toàn".
- **Cấu trúc môi trường điều hòa tất cả.** Môi trường deterministic cho phép đi men mép nguy hiểm mà không tốn gì (policy optimal-but-risky); thêm nhiễu (slippery) sẽ làm chính policy đó trở nên tồi vì xác suất trượt vào cliff — khi đó policy "an toàn" chạy vòng trên mới tối ưu, và $\gamma$ lớn (nhìn xa) trở nên cần thiết để nhìn thấy cái giá đó.

## (b) Vì sao Value Iteration và Policy Iteration cùng đạt được policy tối ưu?

Cả hai đều đang giải **cùng một phương trình**: $V^* = \mathcal{T}V^*$, trong đó $\mathcal{T}$ là toán tử Bellman tối ưu — một **phép co trên không gian hàm giá trị hữu hạn chiều**, nên theo định lý Banach fixed-point, phương trình có **đúng một** nghiệm $V^*$ (và do đó một tập policy tối ưu). Hai thuật toán chỉ là hai đường đi tới điểm bất động duy nhất ấy:

- **VI** áp dụng trực tiếp $\mathcal{T}$: $V_{k+1} = \mathcal{T}V_k$; phép co đảm bảo $V_k \to V^*$ bất kể khởi tạo.
- **PI** đi vòng qua hai phép ánh xạ khác: evaluation $\mathcal{T}^\pi$ (kéo $V$ về $V^\pi$ của policy hiện tại) và improvement (greedy hóa). Dãy $V^{\pi_k}$ tăng đơn điệu và bị chặn bởi $V^*$; nó chỉ dừng khi $\pi = \text{greedy}(V^\pi)$, tức $V^\pi = \mathcal{T}V^\pi$ — đúng phương trình của $V^*$.

Vì nghiệm duy nhất, hai đường hội tụ **phải** gặp nhau: số liệu đo được trên Taxi khẳng định — Mean Value trùng nhau tới 6 chữ số thập phân, `max |V_VI − V_PI| = 0`, `Different actions = 0`. (Khác nhau duy nhất có thể xảy ra là ở các state có nhiều action đồng giá trị tối ưu — tie của `argmax` — chứ không phải khác về chất lượng; ở đây cả hai dùng chung quy tắc tie-break của `np.argmax` nên ngay cả tie cũng trùng.)

# ✅ Kết luận

| Yêu cầu của đề | Hoàn thành ở đâu |
|---|---|
| Value Iteration trên CliffWalking-v0 (TODO 1–3 + comment lý giải) | PART 1 — hàm `value_iteration` |
| Policy Iteration trên Taxi-v3 (TODO 1–3 + comment lý giải) | PART 2 — hàm `policy_iteration` |
| In kết quả cả 2 môi trường (mean value, policy mũi tên) | PART 1 Evaluation · PART 2 Testing |
| Trả lời THINK FIRST / THINK BEFORE CODING / REFLECT | Các mục 🧠 tương ứng |
| Thảo luận: reward & discount shaping behavior | Thí nghiệm 1 & 2 + 📌 Thảo luận (a) |
| Thảo luận: vì sao VI và PI cùng policy tối ưu | 📌 Thảo luận (b) — kèm kiểm chứng bằng số liệu |
| Thí nghiệm mở rộng (γ sweep, penalty sweep, bảng so sánh chi phí, rollout) | PART 1 🧪 · PART 2 bảng so sánh + rollout |

**Kết quả chính (γ = 0.9, θ = 1e-6):**

- **CliffWalking:** VI hội tụ sau 15 sweep; policy men mép cliff (13 bước); mean V ≈ −5.089. Policy **không đổi** khi γ ∈ {0.5, 0.9, 0.99} hay khi phạt cliff đổi −100 → −10 (ngưỡng thay đổi thực ≈ −0.75).
- **Taxi:** VI và PI cho **cùng policy và cùng giá trị** (0 action khác nhau); VI nhanh hơn (~114 so với ~471 lượt backup/state) vì evaluation của PI tốn ~95% chi phí; rollout thực tế cho lộ trình tối ưu (tổng reward +10 và +8).

## 📚 Tài liệu tham khảo

- Sutton & Barto, *Reinforcement Learning: An Introduction*, Chapter 4 (Dynamic Programming).
- Gymnasium Documentation – Environments: <https://gymnasium.farama.org/environments/>
- Pieter Abbeel, *Markov Decision Processes and Exact Solution Methods* (UC Berkeley).